In [2]:
import sys
import os
import json
from pathlib import Path

# 1. Path Setup
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_rq1 import *
print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

# 2. Define File Paths
# Point this directly to your test file
TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "wikidata_testset_RQ1.json") 
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "sparqlce", "models", "wikidata_model_mrr_0.3238.pt") 

TOPKS = [3, 5, 10, 20]
BATCH = 32

# 3. Load the test dataset directly
test_data = load_triplets(TEST_DATA_PATH)

print(f"Dataset Loaded -> Test Split: {len(test_data)}")

# 4. Build corpus and eval items using ONLY the test data
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Evaluation Ready -> Anchors: {len(eval_items)} | Corpus: {len(global_corpus)}")

✓ Success! Project root set to: /upb/users/m/manzoor/profiles/unix/cs/SPARQL-Embeddings-main-refactored
Dataset Loaded -> Test Split: 186
Evaluation Ready -> Anchors: 186 | Corpus: 2229


In [ ]:
# SPARQLCE
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# E5
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')

# Qwen
evaluate_sentence_transformer('Qwen/Qwen2.5-0.5B-Instruct', 'Qwen2.5-0.5B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)

In [3]:
display_name = "SBERT/MiniLM" 
model_id = "sentence-transformers/all-MiniLM-L6-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10365.75it/s]



SBERT/MiniLM
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0968  0.0195  0.0307  0.0959  0.1290    0.0968  0.0908
@5     0.0978  0.0324  0.0457  0.1004  0.1505    0.0978  0.0938
@10    0.0849  0.0569  0.0638  0.1086  0.2043    0.0849  0.0908
@20    0.0462  0.0619  0.0494  0.1096  0.2204    0.0462  0.0705


In [4]:
display_name = "SBERT/MPNet"
model_id = "sentence-transformers/all-mpnet-base-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8047.07it/s]



SBERT/MPNet
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0950  0.0184  0.0298  0.0923  0.1290    0.0950  0.0883
@5     0.1022  0.0337  0.0483  0.0961  0.1452    0.1022  0.0950
@10    0.0860  0.0570  0.0648  0.1062  0.2258    0.0860  0.0898
@20    0.0508  0.0679  0.0547  0.1087  0.2581    0.0508  0.0733
